### 1. Machine Learning

Machine learning (ML) should be used when the goal is to identify patterns or make predictions from data where explicit rule-based programming would be impractical or impossible. Machine Learning is particularly appropriate when dealing with large datasets, high-dimensional features, non-linear relationships, or problems where patterns are too complex for humans to manually encode. For example, predicting survival in the Titanic dataset involves interactions between age, class, fare, and sex. Rather than writing thousands of conditional rules, Machine Learning can learn patterns directly from historical data.

Machine Learning is most effective when:
* There is sufficient high-quality labeled data.
* The problem is predictive rather than purely explanatory.
* Patterns are stable enough to generalize to future data.
* The cost of prediction errors is manageable or measurable.

In summary, Machine Learning is appropriate when prediction, scalability, and pattern detection outweigh the need for simple transparency and deterministic logic.

### 2. Pre-processing on the dataset

**Technique 1: Handling missing values**
Filling continuous null variables with their medians and categorical variables with their modes.

**Technique 2: Feature engineering**
Creating the `family_size` feature to map passenger relationships together.

**Technique 3: One-Hot Encoding**
Converting non-numeric strings into computer-readable matrix columns.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

df = pd.read_csv('titanic.csv')

df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df['Fare'].fillna(df['Fare'].median(), inplace=True)

df['family_size'] = df['SibSp'] + df['Parch'] + 1

df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

df.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1, inplace=True)

### 3. Correlation Matrix with a Heatmap

Sex (male) shows strong negative correlation with survival. Fare and Pclass also show meaningful relationships. Family_size has moderate correlation. Age shows weak correlation after imputation.

**Selected features (≤10):**
* Pclass
* Age
* Fare
* family_size
* Sex_male
* Embarked_Q
* Embarked_S

These provide predictive strength while limiting multicollinearity.

In [2]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

### 4. Box and Whisker Plot for Fare

Null values were replaced with the median fare.

The boxplot shows heavy right skew and significant outliers. Most passengers paid relatively low fares, while a few paid extremely high amounts, likely first-class passengers. This suggests a possible need for log transformation in modeling.

In [3]:
df.boxplot(column='Fare')
plt.show()

### 5. Age Binning with Timers

`pd.cut()` is more performant and vectorized, making it significantly faster on large datasets. Lambda with `apply()` is slower because it operates row-by-row in Python rather than optimized C-level operations.

In [4]:
start = time.time()
df['age_bin_cut'] = pd.cut(
    df['Age'],
    bins=[0,12,20,60,120],
    labels=['Child','Teen','Adult','Senior']
)
print("pd.cut time:", time.time() - start)

### 6. Subplot Histograms of Fares by Age Bin

Adults paid the widest range of fares, including the highest outliers. Children and teens show fewer extreme high fares. Seniors display moderate spread but fewer observations overall. Fare distribution remains right-skewed in all age groups. This suggests socioeconomic status (captured partly by fare and Pclass) likely influenced survival outcomes more strongly than age alone.

In [5]:
fig, axes = plt.subplots(2,2, figsize=(12,8))
bins = ['Child','Teen','Adult','Senior']

for ax, b in zip(axes.flatten(), bins):
    df[df['age_bin_cut']==b]['Fare'].hist(ax=ax)
    ax.set_title(b)

plt.tight_layout()
plt.show()